# BrewBeats — Mini Project 2: Section A (Spark DataFrames — Q3 to Q8)
**Folder:** `Section A -  Spark RDD & DataFrames (Databricks) and Local Spark for RDD/`  
**Stack:** Apache Spark (PySpark DataFrames & Spark SQL)

---
## Section A - Q3: Load `coffee_sales.csv` into a DataFrame and find the total revenue.

In [ ]:
# === Section A - Q3: Load coffee_sales.csv into a DataFrame and find the total revenue ===

# SparkSession is our main entry point (the 'on switch') to work with Spark DataFrames.
# Unlike RDDs (which just run our Python code step by step), DataFrames use Spark's Catalyst Optimizer —
# Spark's built-in 'smart brain' that reads our query and automatically finds the fastest way to execute it.
# We also import functions as 'F' so we can use Spark's built-in math functions (like F.sum, F.count, F.round)
# without mixing them up with standard Python functions.
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Start our Spark session:
# - .appName() gives a name to our Spark job
# - .master("local[*]") tells Spark to run locally on my laptop using all available CPU cores [*]
# - .getOrCreate() starts a new session or reuses the one already running
spark = (
    SparkSession.builder
    .appName("BrewBeats_MiniProject2_DataFrames")
    .master("local[*]")
    .getOrCreate()
)

# Hide the long red INFO/WARN logs from Spark so the output stays clean
spark.sparkContext.setLogLevel("ERROR")

# Load coffee_sales.csv into a DataFrame:
# - header=True tells Spark that the first row has the column names (date, datetime, cash_type, card, money, coffee_name)
# - inferSchema=True tells Spark to automatically detect the data types (so 'money' is read as a number instead of text)
coffee_df = spark.read.csv("coffee_sales.csv", header=True, inferSchema=True)

print("=== Section A - Q3: Load coffee_sales.csv into a DataFrame & Find Total Revenue ===")
print(f"Total Rows Loaded: {coffee_df.count()}\n")

# Show the first 5 rows (truncate=False prevents Spark from cutting off long timestamps with '...')
print("Sample Records (Top 5):")
coffee_df.show(5, truncate=False)

# Calculate total revenue using .agg() (aggregate), which summarizes all 3,636 rows into 1 summary row:
# - F.count("*") counts the total number of transactions (rows)
# - F.sum("money") adds up all values in the 'money' column to get total revenue
# - F.avg("money") calculates the average amount spent per transaction
# - F.round(..., 2) rounds the decimals to 2 places (cents)
# - .alias("...") renames the result columns (just like AS in SQL)
revenue_summary_df = coffee_df.agg(
    F.count("*").alias("total_transactions"),
    F.round(F.sum("money"), 2).alias("total_revenue"),
    F.round(F.avg("money"), 2).alias("avg_transaction_value")
)

print("Total Revenue Summary:")
revenue_summary_df.show()

# .collect()[0]["total_revenue"] pulls the number out of the 1-row Spark table into a normal Python variable
# so we can print it nicely formatted with a dollar sign
total_revenue = revenue_summary_df.collect()[0]["total_revenue"]
print(f"Total Revenue: ${total_revenue:,.2f}")

=== Section A - Q3: Load coffee_sales.csv into a DataFrame & Find Total Revenue ===
Total Rows Loaded: 3636

Sample Records (Top 5):
+----------+-----------------------+---------+-------------------+-----+-------------+
|      date|               datetime|cash_type|               card|money|  coffee_name|
+----------+-----------------------+---------+-------------------+-----+-------------+
|2024-03-01| 2024-03-01 10:15:50.52|     card|ANON-0000-0000-0001| 38.7|        Latte|
|2024-03-01|2024-03-01 12:19:22.539|     card|ANON-0000-0000-0002| 38.7|Hot Chocolate|
|2024-03-01|2024-03-01 12:20:18.089|     card|ANON-0000-0000-0002| 38.7|Hot Chocolate|
|2024-03-01|2024-03-01 13:46:33.006|     card|ANON-0000-0000-0003| 28.9|    Americano|
|2024-03-01|2024-03-01 13:48:14.626|     card|ANON-0000-0000-0004| 38.7|        Latte|
+----------+-----------------------+---------+-------------------+-----+-------------+
only showing top 5 rows
Total Revenue Summary:
+------------------+-------------+---

---
## Section A - Q4: Find the number of transactions for each `coffee_name` using `groupBy`.

In [2]:
# === Section A - Q4: Find the number of transactions for each coffee_name using groupBy ===

# Group the DataFrame by 'coffee_name' (just like GROUP BY coffee_name in SQL) so all sales
# of the same drink are placed in the same bucket.
# Then use .agg() to calculate summary metrics for each coffee type:
# - F.count("*") counts how many transactions (cups sold) belong to each coffee_name
# - F.round(F.sum("money"), 2) adds up the total revenue for each coffee_name, rounded to 2 decimals
# - F.round(F.avg("money"), 2) finds the average price per cup for that coffee_name
# Finally, .orderBy(F.col("transaction_count").desc()) sorts the table from most sold to least sold.
transactions_by_coffee_df = (
    coffee_df
    .groupBy("coffee_name")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("money"), 2).alias("total_revenue"),
        F.round(F.avg("money"), 2).alias("avg_price")
    )
    .orderBy(F.col("transaction_count").desc())
)

# Display the results table (truncate=False ensures full coffee names like 'Americano with Milk' aren't cut off)
print("=== Section A - Q4: Number of Transactions for Each coffee_name (groupBy) ===")
transactions_by_coffee_df.show(truncate=False)

=== Section A - Q4: Number of Transactions for Each coffee_name (groupBy) ===
+-------------------+-----------------+-------------+---------+
|        coffee_name|transaction_count|total_revenue|avg_price|
+-------------------+-----------------+-------------+---------+
|Americano with Milk|              824|     25269.12|    30.67|
|              Latte|              782|      27866.3|    35.63|
|          Americano|              578|     15062.26|    26.06|
|         Cappuccino|              501|     18034.14|     36.0|
|            Cortado|              292|      7534.86|     25.8|
|      Hot Chocolate|              282|     10172.46|    36.07|
|              Cocoa|              243|      8678.16|    35.71|
|           Espresso|              134|      2814.28|     21.0|
+-------------------+-----------------+-------------+---------+


---
## Section A - Q5: Add a new column for month from the `date` column, and find total revenue per month.

In [3]:
# === Section A - Q5: Add a new column for month from the date column, and find total revenue per month ===

# Use .withColumn() to create new columns derived from the 'date' column:
# - F.date_format(F.col("date"), "yyyy-MM") extracts Year-Month (e.g., '2024-03') so March 2024 and March 2025 stay separate
# - F.month(F.col("date")) extracts the month number (1 to 12)
# - F.date_format(F.col("date"), "MMMM") extracts the full month name (e.g., 'March', 'October')
coffee_with_month_df = (
    coffee_df
    .withColumn("year_month", F.date_format(F.col("date"), "yyyy-MM"))
    .withColumn("month", F.month(F.col("date")))
    .withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
)

# Group by the month columns and sum 'money' to get the total revenue for each month:
# - F.count("*") counts how many coffee sales happened in that month
# - F.round(F.sum("money"), 2) adds up the total revenue for that month, rounded to 2 decimal places
# - .orderBy("year_month") orders the table chronologically from earliest month to latest month
monthly_revenue_df = (
    coffee_with_month_df
    .groupBy("year_month", "month", "month_name")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("money"), 2).alias("monthly_revenue")
    )
    .orderBy("year_month")
)

print("=== Section A - Q5: Add Month Column & Find Total Revenue per Month ===")
monthly_revenue_df.show(20, truncate=False)

=== Section A - Q5: Add Month Column & Find Total Revenue per Month ===
+----------+-----+----------+-----------------+---------------+
|year_month|month|month_name|transaction_count|monthly_revenue|
+----------+-----+----------+-----------------+---------------+
|   2024-03|    3|     March|              206|         7050.2|
|   2024-04|    4|     April|              196|        6720.56|
|   2024-05|    5|       May|              267|        9063.42|
|   2024-06|    6|      June|              227|        7758.76|
|   2024-07|    7|      July|              237|        6915.94|
|   2024-08|    8|    August|              272|        7613.84|
|   2024-09|    9| September|              344|        9988.64|
|   2024-10|   10|   October|              426|       13891.16|
|   2024-11|   11|  November|              259|        8590.54|
|   2024-12|   12|  December|              259|        8237.74|
|   2025-01|    1|   January|              201|        6398.86|
|   2025-02|    2|  February|   

---
## Section A - Q6: In 3-4 lines, explain the difference between an RDD and a DataFrame.

1. **Structure:** An **RDD** is a low-level collection of raw, unstructured items with no column names, whereas a **DataFrame** organizes data into a structured table with named columns and data types (a schema).
2. **How We Write Code:** With an **RDD**, we tell Spark *how* to process data step-by-step using Python `lambda` functions (`.map()`, `.reduceByKey()`) and column index numbers (`cols[16]`).
3. **Ease of Use:** With a **DataFrame**, we simply tell Spark *what* we want using SQL-like table operations by column name (`.groupBy("coffee_name")`, `F.sum("money")`).
4. **Performance:** Because a **DataFrame** has a schema, Spark's **Catalyst Optimizer** automatically builds the fastest execution plan, making DataFrames much faster and more memory-efficient than Python RDDs.

---
## Section A - Q7: How many times the songs had ‘love’ in them?

In [4]:
# === Section A - Q7: How many times the songs had 'love' in them? ===

# 1. Load 'Spotify_data.csv' into a Spark DataFrame with column names (header=True) and data types (inferSchema=True)
spotify_df = spark.read.csv("Spotify_data.csv", header=True, inferSchema=True)

# 2. Filter for songs that have the word 'love' in their title:
#    - F.lower(F.col("song_title")) converts every song title to lowercase first so we catch 'Love', 'LOVE', and 'love'
#    - .contains("love") checks if the word 'love' appears anywhere inside the title (just like ILIKE '%love%' in SQL)
#    - .select(...) picks the main columns we want to preview in our table
love_songs_df = (
    spotify_df
    .filter(F.lower(F.col("song_title")).contains("love"))
    .select("song_title", "artist", "danceability", "energy", "valence")
)

# 3. Count the matching rows with .count() and show a sample of 15 songs
love_songs_count = love_songs_df.count()

print("=== Section A - Q7: How Many Times Songs Had 'love' in Them ===")
print(f"Total Songs with 'love' in the Title (Case-Insensitive): {love_songs_count}\n")
print("Sample of 15 Songs Containing 'love':")
love_songs_df.show(15, truncate=False)

=== Section A - Q7: How Many Times Songs Had 'love' in Them ===
Total Songs with 'love' in the Title (Case-Insensitive): 88

Sample of 15 Songs Containing 'love':
+---------------------------------------------+----------------------+------------+------+-------+
|                                   song_title|                artist|danceability|energy|valence|
+---------------------------------------------+----------------------+------------+------+-------+
|                      House of Jealous Lovers|           The Rapture|       0.747|  0.74|   0.77|
|                        If I Gave You My Love|             Myron & E|       0.512| 0.564|  0.401|
|        Who Be Lovin Me (feat. ILOVEMAKONNEN)|             Santigold|       0.805| 0.687|  0.683|
|Ain't No Love In The Heart Of The City - S...|"Bobby ""Blue"" Bland"|       0.559| 0.656|  0.671|
|                                     Our Love|               Caribou|       0.726| 0.497|  0.472|
|                        Have Love Will Trave

---
## Section A - Q8: BrewBeats wants to play some party songs for their 5th Birthday, curate a songs list for them.

In [5]:
# === Section A - Q8: Curate a 5th Birthday party songs playlist for BrewBeats ===

# To build a great birthday party playlist, we filter for upbeat, feel-good dance tracks using Spotify's audio features:
# - danceability > 0.80 : very easy and fun to dance to (scale is 0.0 to 1.0)
# - energy > 0.80       : fast, loud, and lively party vibe (scale is 0.0 to 1.0)
# - valence > 0.75      : happy, cheerful, positive mood (high valence = happy; low valence = sad)
# - tempo >= 115        : upbeat tempo in Beats Per Minute (BPM)
#
# We also create a custom 'party_score' column (the average of danceability, energy, and valence)
# using .withColumn() so we can rank the best party songs from highest score to lowest!
party_playlist_df = (
    spotify_df
    .filter(
        (F.col("danceability") > 0.80) &
        (F.col("energy") > 0.80) &
        (F.col("valence") > 0.75) &
        (F.col("tempo") >= 115)
    )
    .withColumn(
        "party_score",
        F.round((F.col("danceability") + F.col("energy") + F.col("valence")) / 3, 3)
    )
    .select("song_title", "artist", "danceability", "energy", "valence", "tempo", "party_score")
    .orderBy(F.col("party_score").desc(), F.col("danceability").desc())
)

print("=== Section A - Q8: BrewBeats 5th Birthday Party Playlist ===")
print(f"Total High-Energy Party Songs Found: {party_playlist_df.count()}\n")
print("Top 15 Curated Party Songs for BrewBeats' 5th Birthday:")
party_playlist_df.show(15, truncate=False)

=== Section A - Q8: BrewBeats 5th Birthday Party Playlist ===
Total High-Energy Party Songs Found: 15

Top 15 Curated Party Songs for BrewBeats' 5th Birthday:
+--------------------------------------+------------------+------------+------+-------+-------+-----------+
|                            song_title|            artist|danceability|energy|valence|  tempo|party_score|
+--------------------------------------+------------------+------------+------+-------+-------+-----------+
|                     My Favorite Snack|Imagination Movers|        0.89| 0.891|  0.947|119.993|      0.909|
|                  Sippin On Some Syrup|     Three 6 Mafia|       0.944| 0.839|  0.891|136.019|      0.891|
|                          Hoochie Mama|       2 LIVE CREW|       0.895| 0.883|  0.893|131.644|       0.89|
|           P.Y.T. (Pretty Young Thing)|   Michael Jackson|       0.888| 0.815|   0.96|127.273|      0.888|
|            Got To Give It Up (Part 1)|       Marvin Gaye|       0.856| 0.874|   0.9